# AI 조직 영역 마스크 테스트

프로덕션에서 쓰는 `tissue_mask` 로직 ([app/routers/ai.py](backend/app/routers/ai.py) 의 `_create_tissue_mask`) 을 프로젝트 밖에서 돌려 시각 검증한다.

- **Method A — 간단형**: 그레이스케일 → 역 Otsu → close/open. 데스크톱 레거시에서 쓰던 방식 (코드는 삭제됐지만 비교 기준선으로 포함).
- **Method B — 정교형** (실사용): H-DAB color deconvolution → Hem/DAB 채널 Otsu + 국소 std(texture) Otsu 합집합 → 확실한 배경 제거 → 외곽 컨투어 fill.

두 방식의 커버리지/IoU 를 나란히 놓고 Method B 가 조직을 얼마나 더 잡아내는지 (또는 과잉 포함하는지) 확인.

In [16]:
# ── OpenSlide DLL 경로 설정 (import 전에) ──
import os
from pathlib import Path

path_project_root = Path.cwd().parent
list_dll_dirs = [
    path_project_root / 'libs' / 'openslide_lib' / 'bin',
    path_project_root / 'libs',
]
for path_dir in list_dll_dirs:
    if path_dir.exists():
        os.environ['OPENSLIDE_PATH'] = str(path_dir)
        os.environ['PATH'] = str(path_dir) + os.pathsep + os.environ.get('PATH', '')
        try:
            os.add_dll_directory(str(path_dir))
        except (AttributeError, OSError):
            pass
        print(f'DLL 등록: {path_dir}')
        break
else:
    print('WARN: openslide DLL 경로 미발견')

DLL 등록: c:\Users\liive\OneDrive\Desktop\project\libs\openslide_lib\bin


In [17]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import openslide
from PIL import Image

print(f'openslide C lib : {openslide.__library_version__}')
print(f'opencv          : {cv2.__version__}')

openslide C lib : 4.0.0
opencv          : 4.13.0


In [18]:
# ── 슬라이드 후보 자동 탐색 ──
path_uploads = path_project_root / 'Mediauto-studio_saas' / 'backend' / 'uploads'
SET_EXT = {'.svs', '.ndpi', '.vms', '.vmu', '.scn', '.mrxs', '.tiff', '.tif'}

list_slides = sorted([
    p for p in path_uploads.rglob('*')
    if p.is_file() and p.suffix.lower() in SET_EXT
])
print(f'슬라이드: {len(list_slides)}개 (상위 15개)')
for i, p in enumerate(list_slides[:15]):
    print(f'  [{i}] {p.relative_to(path_uploads)}')

# 바꾸고 싶으면 int_pick 만 수정
int_pick = 6
path_slide = list_slides[int_pick] if list_slides else None
print(f'\n선택: {path_slide}')

슬라이드: 19개 (상위 15개)
  [0] HnE(BR)\CODIPAI-BRCA-SS-00194-S-TP-01.svs
  [1] HnE(BR)\CODIPAI-BRCA-SS-00198-S-TP-01.svs
  [2] HnE(ST)\CODIPAI-STBX-SS-04353-S-EB-01.ndpi
  [3] HnE(ST)\CODIPAI-STBX-SS-04354-S-EB-01.ndpi
  [4] HnE(ST)\CODIPAI-STBX-SS-04358-S-EB-01.ndpi
  [5] HnE(ST)\CODIPAI-STOP-SS-01376-S-TP-01.ndpi
  [6] IHC(ER,PR)\CODIPAI-BRCA-SS-00194-I-PR-01.svs
  [7] IHC(ER,PR)\CODIPAI-BRCA-SS-00195-I-PR-01.svs
  [8] IHC(ER,PR)\CODIPAI-BRCA-SS-00196-I-ER-01.svs
  [9] IHC(ER,PR)\CODIPAI-BRCA-SS-00197-I-ER-01.svs
  [10] IHC(HER2)\CODIPAI-BRCA-SS-00192-I-HR-01.svs
  [11] IHC(HER2)\CODIPAI-BRCA-SS-00194-I-HR-01.svs
  [12] IHC(HER2)\CODIPAI-BRCA-SS-00198-I-HR-01.svs
  [13] IHC(KI-67)\CODIPAI-BRCA-SS-00192-I-KI-01.svs
  [14] IHC(KI-67)\CODIPAI-BRCA-SS-00194-I-KI-01.svs

선택: c:\Users\liive\OneDrive\Desktop\project\Mediauto-studio_saas\backend\uploads\IHC(ER,PR)\CODIPAI-BRCA-SS-00194-I-PR-01.svs


In [19]:
assert path_slide is not None
obj_slide = openslide.OpenSlide(str(path_slide))

print(f'file   : {path_slide.name}')
print(f'vendor : {obj_slide.properties.get("openslide.vendor", "?")}')
print(f'dims   : {obj_slide.dimensions}')
print(f'mpp    : {obj_slide.properties.get("openslide.mpp-x", "?")}')
print(f'obj pow: {obj_slide.properties.get("openslide.objective-power", "?")}')

file   : CODIPAI-BRCA-SS-00194-I-PR-01.svs
vendor : aperio
dims   : (85106, 82076)
mpp    : 0.26258300000000001
obj pow: 40


## 1. 썸네일 raw vs gamma 0.5 비교

`v_out = 255 * (v_in/255) ^ γ`.
γ < 1 이면 어두운 톤이 들려 옅은 조직이 Otsu 에서 더 잘 잡힌다.

In [ ]:
FLOAT_GAMMA = 0.5
INT_DOWNSAMPLE = 128


def _gamma_lut(float_gamma: float) -> np.ndarray:
    # 256-entry uint8 LUT — v_out = 255 * (v_in/255) ^ γ
    np_x = np.arange(256, dtype=np.float32) / 255.0
    np_y = np.clip(np.power(np_x, float_gamma) * 255.0, 0, 255)
    return np_y.astype(np.uint8)


def apply_gamma(np_img: np.ndarray, float_gamma: float) -> np.ndarray:
    # HxWxC uint8 에 채널 공통 γ 적용
    if abs(float_gamma - 1.0) < 1e-6:
        return np_img
    return _gamma_lut(float_gamma)[np_img]


# 썸네일 한 번만 꺼내서 두 변형 생성
obj_thumb = obj_slide.get_thumbnail((
    obj_slide.dimensions[0] // INT_DOWNSAMPLE,
    obj_slide.dimensions[1] // INT_DOWNSAMPLE,
))
np_thumb_raw = np.array(obj_thumb)[..., :3]
np_thumb_gamma = apply_gamma(np_thumb_raw, FLOAT_GAMMA)

fig, list_ax = plt.subplots(1, 2, figsize=(14, 5))
list_ax[0].imshow(np_thumb_raw);   list_ax[0].set_title(f'raw thumbnail  {np_thumb_raw.shape[:2]}');     list_ax[0].axis('off')
list_ax[1].imshow(np_thumb_gamma); list_ax[1].set_title(f'gamma {FLOAT_GAMMA} applied');                list_ax[1].axis('off')
plt.tight_layout(); plt.show()

print(f'raw   mean RGB : {np_thumb_raw.reshape(-1,3).mean(0).round(1)}')
print(f'gamma mean RGB : {np_thumb_gamma.reshape(-1,3).mean(0).round(1)}')

## 2. 감마 적용 썸네일에서 조직 마스크 추출

프로덕션 (`app/routers/ai.py` 의 `_create_tissue_mask`) 과 동일한 H-DAB 파이프라인을
**raw 썸네일** 과 **γ=0.5 썸네일** 각각에 적용해, 커버리지와 오버레이로 차이를 본다.

In [ ]:
def tissue_mask_hdab_from_thumb(np_rgb: np.ndarray, tuple_slide_dims):
    # app/routers/ai.py 의 _create_tissue_mask 와 동일한 H-DAB 파이프라인.
    # 입력 썸네일(np_rgb) 은 이미 원하는 전처리(γ 등)가 끝난 상태여야 한다.
    np_sm = np.array([
        [0.650, 0.704, 0.286],
        [0.268, 0.570, 0.776],
        [0.711, 0.423, 0.500],
    ], dtype=np.float64)
    np_sm = np_sm / np.linalg.norm(np_sm, axis=1, keepdims=True)
    np_dc = np.linalg.inv(np_sm)

    int_h, int_w = np_rgb.shape[:2]
    np_rgb_f = np.maximum(np_rgb.astype(np.float64), 1.0)
    np_od = -np.log(np_rgb_f / 255.0)
    np_stain_od = (np_od.reshape(-1, 3) @ np_dc.T).reshape(int_h, int_w, 3)

    np_hem = np.clip(np_stain_od[:, :, 0], 0, None)
    np_dab = np.clip(np_stain_od[:, :, 1], 0, None)
    float_hm = max(np.percentile(np_hem, 99.5), 0.01)
    float_dm = max(np.percentile(np_dab, 99.5), 0.01)
    np_hem_u8 = np.clip(np_hem / float_hm * 255, 0, 255).astype(np.uint8)
    np_dab_u8 = np.clip(np_dab / float_dm * 255, 0, 255).astype(np.uint8)
    _, np_hm_mask = cv2.threshold(np_hem_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, np_dm_mask = cv2.threshold(np_dab_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    np_gray = cv2.cvtColor(np_rgb, cv2.COLOR_RGB2GRAY)
    np_g_f = np_gray.astype(np.float32)
    np_lm = cv2.blur(np_g_f, (15, 15))
    np_lsq = cv2.blur(np_g_f ** 2, (15, 15))
    np_std = np.sqrt(np.maximum(np_lsq - np_lm ** 2, 0))
    np_std_u8 = np.clip(np_std * 10, 0, 255).astype(np.uint8)
    _, np_tm = cv2.threshold(np_std_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    np_hist = cv2.calcHist([np_gray], [0], None, [256], [0, 256]).flatten()
    int_bg_peak = int(np.argmax(np_hist[128:]) + 128)
    np_bg = (np_gray >= int(int_bg_peak * 0.95))

    np_union = (np_hm_mask > 0) | (np_dm_mask > 0) | (np_tm > 0)
    np_union[np_bg] = False
    np_mask = (np_union.astype(np.uint8)) * 255
    np_mask = cv2.morphologyEx(np_mask, cv2.MORPH_CLOSE, np.ones((25, 25), np.uint8))
    np_mask = cv2.morphologyEx(np_mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

    contours, _ = cv2.findContours(np_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    np_filled = np.zeros_like(np_mask)
    int_min_area = max(int(int_h * int_w * 0.0005), 50)
    for cnt in contours:
        if cv2.contourArea(cnt) < int_min_area:
            continue
        cv2.drawContours(np_filled, [cnt], -1, 255, thickness=cv2.FILLED)

    int_tw = tuple_slide_dims[0] // 64
    int_th = tuple_slide_dims[1] // 64
    return cv2.resize(np_filled, (int_tw, int_th), interpolation=cv2.INTER_NEAREST)


def _overlay(np_thumb_bg, np_mask, color_rgb=(0, 180, 255), float_alpha=0.45):
    int_h, int_w = np_thumb_bg.shape[:2]
    np_m_resized = cv2.resize(np_mask, (int_w, int_h), interpolation=cv2.INTER_NEAREST)
    np_color = np.zeros_like(np_thumb_bg)
    np_color[..., 0] = color_rgb[0]
    np_color[..., 1] = color_rgb[1]
    np_color[..., 2] = color_rgb[2]
    np_m_bool = (np_m_resized > 0)[..., None]
    return np.where(
        np_m_bool,
        (np_thumb_bg * (1 - float_alpha) + np_color * float_alpha).astype(np.uint8),
        np_thumb_bg,
    )


# ── 두 썸네일 각각에서 마스크 추출 ──
tuple_dims = obj_slide.dimensions
np_mask_raw   = tissue_mask_hdab_from_thumb(np_thumb_raw,   tuple_dims)
np_mask_gamma = tissue_mask_hdab_from_thumb(np_thumb_gamma, tuple_dims)

float_cov_raw   = (np_mask_raw   > 0).mean() * 100
float_cov_gamma = (np_mask_gamma > 0).mean() * 100

# IoU / 추가 포함 영역
np_bin_raw   = np_mask_raw   > 0
np_bin_gamma = np_mask_gamma > 0
float_iou = (np_bin_raw & np_bin_gamma).sum() / max((np_bin_raw | np_bin_gamma).sum(), 1)
float_gamma_only = (np_bin_gamma & ~np_bin_raw).sum() / max(np_bin_gamma.sum(), 1)

print(f'coverage raw    : {float_cov_raw:.2f}%')
print(f'coverage gamma  : {float_cov_gamma:.2f}%   (Δ {float_cov_gamma - float_cov_raw:+.2f}%)')
print(f'IoU(raw, gamma) : {float_iou * 100:.2f}%')
print(f'gamma 전용 포함 : {float_gamma_only * 100:.2f}%  (γ 덕에 새로 잡힌 비율)')

# 오버레이는 raw 썸네일 위에 그대로 그림 — 시각 비교 일관성
np_ov_raw   = _overlay(np_thumb_raw, np_mask_raw,   color_rgb=(  0, 180, 255))
np_ov_gamma = _overlay(np_thumb_raw, np_mask_gamma, color_rgb=(255,  80, 180))

fig, list_ax = plt.subplots(1, 2, figsize=(14, 6))
list_ax[0].imshow(np_ov_raw);   list_ax[0].set_title(f'mask from raw thumb  cov={float_cov_raw:.1f}%');             list_ax[0].axis('off')
list_ax[1].imshow(np_ov_gamma); list_ax[1].set_title(f'mask from γ={FLOAT_GAMMA} thumb  cov={float_cov_gamma:.1f}%'); list_ax[1].axis('off')
plt.tight_layout(); plt.show()

## 3. γ 스윕 — 어느 값이 최적인가

`v_out = 255 * (v_in/255) ^ γ`
- γ > 1 → 어둡게 → 배경 피크가 낮아져 `definite_bg` 제외 덜 공격적 → tissue 더 포함
- γ < 1 → 밝게 → 반대 경향

일반적으로 γ ∈ [1.5, 2.5] 에서 옅은 조직까지 포획 + 노이즈는 억제.

In [ ]:
list_gammas = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]

fig, list_ax = plt.subplots(2, 3, figsize=(18, 11))
for int_i, float_g in enumerate(list_gammas):
    np_thumb_gx = apply_gamma(np_thumb_raw, float_g)
    np_mask_gx = tissue_mask_hdab_from_thumb(np_thumb_gx, tuple_dims)
    float_cov = (np_mask_gx > 0).mean() * 100

    # 오버레이는 raw 썸네일 위에 — 시각 비교 일관성 유지
    np_ov = _overlay(np_thumb_raw, np_mask_gx, color_rgb=(0, 200, 100))
    ax = list_ax[int_i // 3, int_i % 3]
    ax.imshow(np_ov)
    ax.set_title(f'γ = {float_g}   coverage = {float_cov:.2f}%', fontsize=11)
    ax.axis('off')
    print(f'γ={float_g:>4}  coverage={float_cov:6.2f}%')

plt.tight_layout(); plt.show()